# BIRDSAI Data Exploration

Run the data pipeline before using this notebook. It summarizes frame manifests, object distributions, split balance, validation issues, and generated annotation checks.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

OUTPUT_ROOT = Path.home() / 'computer_vision' / 'outputs'
MANIFEST_ROOT = OUTPUT_ROOT / 'manifests'
print('Output root:', OUTPUT_ROOT)

In [ ]:
train_frames = pd.read_csv(MANIFEST_ROOT / 'trainreal_frames.csv')
test_frames = pd.read_csv(MANIFEST_ROOT / 'testreal_frames.csv')
sequences = pd.read_csv(MANIFEST_ROOT / 'sequences.csv')
issues = pd.read_csv(MANIFEST_ROOT / 'annotation_issues.csv')

summary = sequences.groupby('split').agg(
    sequences=('sequence_id', 'count'),
    frames=('num_frames', 'sum'),
    objects=('num_objects', 'sum'),
    animals=('num_animals', 'sum'),
    humans=('num_humans', 'sum'),
    small=('num_small', 'sum'),
    medium=('num_medium', 'sum'),
    large=('num_large', 'sum'),
)
display(summary)
print('Validation issues:', len(issues))
if not issues.empty:
    display(issues['issue'].value_counts().to_frame('count'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
summary[['animals', 'humans']].plot.bar(ax=axes[0], title='Class distribution')
summary[['small', 'medium', 'large']].plot.bar(ax=axes[1], title='Object-size distribution')
for axis in axes:
    axis.set_ylabel('Bounding boxes')
    axis.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
split_report_path = OUTPUT_ROOT / 'splits' / 'split_report.json'
split_report = json.loads(split_report_path.read_text())
split_table = pd.DataFrame({
    name: values for name, values in split_report.items()
    if name in {'train', 'validation', 'test'}
}).T
display(split_table)

In [ ]:
figure_paths = sorted((OUTPUT_ROOT / 'figures').glob('*/*.jpg'))[:12]
if not figure_paths:
    print('No annotation-check images found.')
else:
    columns = 3
    rows = (len(figure_paths) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(15, 5 * rows))
    axes = axes.ravel() if hasattr(axes, 'ravel') else [axes]
    for axis, path in zip(axes, figure_paths):
        axis.imshow(Image.open(path))
        axis.set_title(path.parent.name + '/' + path.name, fontsize=8)
        axis.axis('off')
    for axis in axes[len(figure_paths):]:
        axis.axis('off')
    plt.tight_layout()
    plt.show()